# Task Statement 1.1: Design and Implement Agentic Loops

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 1: Agentic Architecture & Orchestration
- **Domain Weight**: 27% of total CCAR-F exam score (highest-weighted domain on the exam)
- **Scenario Alignment**: Scenario 1 (Customer Support) & Scenario 4 (Developer Productivity)

### The Core Goal
To master the construction of autonomous, multi-turn agentic loops using Claude Messages API primitives. An architect must know how to inspect API response metadata deterministically, maintain bidirectional conversation state with tool execution blocks, manage emergency brakes, and prevent runaway iteration cycles.

### Why This Matters in Production
In traditional software, execution flows along rigid procedural branches. In agentic systems, the LLM acts as an autonomous runtime engine that decides *when* to gather information, *which* external tools to invoke, and *when* it has amassed sufficient facts to conclude its reasoning. 
If an orchestrator relies on fragile string parsing (e.g. checking whether Claude wrote `"Task finished"`), the loop will silently hang, abort prematurely, or hallucinate completion when Claude uses slightly different phrasing. In production, building the loop correctly prevents runaway costs, infinite tool recursion, and lost context.

### Exam Importance
Task Statement 1.1 is tested heavily because it represents the fundamental execution engine of all Claude agents. Exam scenarios will repeatedly test your ability to reject naive completion checks, understand the exact two-turn tool execution contract, and enforce safety boundaries without breaking agent autonomy.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Problem: The Naive Loop Fallacy
Developers new to the Claude API often try to control agent loops using prompt instructions and text analysis:
```python
# ❌ FRAGILE ANTI-PATTERN: String parsing loop termination
response = client.messages.create(...)
if "I have finished" in response.content[0].text or "Done" in response.content[0].text:
    break
```
**Why this fails in production**:
1. **Unpredictable Phrasing**: Claude might say `"Here is the completed analysis"`, `"I've gathered all required facts"`, or simply present the final Markdown table without using the magic word.
2. **Intermediate Mentions**: Claude might write `"I am not done yet; I will now call the calculator"`—a naive string check on `"done"` could trigger false-positive termination.
3. **Structured Tool Use is Ignored**: Claude signals intent to call tools through structured metadata (`stop_reason == "tool_use"`), not through chat text.

### 2. The Architectural Solution: Deterministic State Handshake
Instead of guessing, production architectures use the **Deterministic Handshake Lifecycle**:
1. **Request Phase**: The host application sends conversational history and a list of tool definitions to Claude (`client.messages.create()`).
2. **Evaluation Phase**: The orchestrator inspects `response.stop_reason`.
3. **Branch 1 (`stop_reason == "tool_use"`)**:
   - Claude has determined it cannot answer without external facts.
   - Claude emits one or more `tool_use` blocks containing tool names and validated input arguments.
   - **Crucial Turn Append Requirement**: The orchestrator must first append the assistant's turn (containing the `tool_use` blocks and any reasoning/thinking blocks) to the message list.
   - The host application executes the requested tools locally (database queries, calculators, web search).
   - The orchestrator appends a `user` turn containing matching `tool_result` blocks referencing the corresponding `tool_use_id`.
   - The loop repeats by sending the updated message list back to Claude.
4. **Branch 2 (`stop_reason == "end_turn"`)**:
   - Claude has determined that it has accumulated all necessary information and has generated its final user-facing response.
   - The loop terminates deterministically.
5. **Emergency Brake (`MAX_ITERATIONS`)**:
   - If a misbehaving tool continuously returns errors or if Claude gets trapped in circular tool calls, an iteration cap (e.g. `MAX_ITERATIONS = 15`) prevents infinite loops and catastrophic token spend.

---

## 🔑 Key Technical Terminology Explained

- **`stop_reason`**: An enumerated string returned by the Claude API indicating why generation ceased:
  - `"tool_use"`: The model stopped to request execution of one or more tools. The client *must* execute tools and return results.
  - `"end_turn"`: The model naturally finished its response. No further tools needed.
  - `"max_tokens"`: The response hit the `max_tokens` ceiling before finishing. Requires token budget tuning.
  - `"stop_sequence"`: A user-specified custom stop string was encountered.
- **Dual Turn History Contract**: The Anthropic API strictly requires that every `tool_result` message from the user is preceded by an assistant message containing the matching `tool_use` blocks. Appending `tool_result` without the preceding `tool_use` turn results in an immediate HTTP `400 Bad Request` API error.
- **`tool_use_id`**: A unique alphanumeric identifier (e.g. `toolu_01AbC...`) assigned to each tool call. The client's `tool_result` block must include this exact `tool_use_id` to correlate input arguments with execution output.
- **Circuit Breaker (`MAX_ITERATIONS`)**: An emergency counter that throws a runtime exception if the loop fails to terminate naturally. It is a safeguard against bugs, **never** the normal condition for ending a task.

---

## 🏗️ Architecture: The Deterministic Loop Lifecycle

```mermaid
flowchart TD
    Start["User Prompt Injected"] --> SendRequest["Call Claude Messages API<br>(model: claude-sonnet-4-5)"]
    SendRequest --> CheckStop{"Inspect response.stop_reason"}
    
    CheckStop -->|"stop_reason == 'tool_use'"| AppendAssistant["1. Append Assistant Turn<br>(Include thinking & tool_use blocks)"]
    AppendAssistant --> ExecLocal["2. Execute Requested Tools Locally<br>(Run calculator, DB query, API)"]
    ExecLocal --> AppendUser["3. Append User Turn<br>(type: tool_result with tool_use_id)"]
    AppendUser --> SendRequest
    
    CheckStop -->|"stop_reason == 'end_turn'"| Terminate["4. Task Complete<br>Extract & Return Final Text"]
    CheckStop -->|"stop_reason == 'max_tokens'"| HandleMaxTokens["5. Token Exhaustion Handler<br>Increase max_tokens or trim prompt"]
    CheckStop -->|"iteration >= MAX_ITERATIONS"| TripCircuit["Emergency Brake Tripped<br>Raise RunawayLoopException"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Event / Mechanism | Correct Architectural Implementation | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Loop Termination** | Rely exclusively on `response.stop_reason == "end_turn"`. | **TRAP**: Parsing text for `"Done"`, `"Completed"`, or using regex on content. | LLMs vary phrasing naturally. Text parsing leads to premature aborts or infinite loops. |
| **Tool Turn History** | Append the assistant message (with `tool_use` blocks), then append user message (with `tool_result` blocks). | **TRAP**: Appending only the tool result directly to history without the assistant's request. | The Anthropic API schema rejects orphan tool results with an HTTP `400 Bad Request`. |
| **Thinking Continuity** | Echo assistant `thinking` blocks back **unmodified with their signature** in subsequent turns. | **TRAP**: Stripping thinking blocks from intermediate turns to "save tokens". | Stripping thinking blocks breaks the cryptographic signature validation and triggers HTTP `400 Bad Request`. |
| **Emergency Bounds** | Set `MAX_ITERATIONS = 15` as an emergency circuit breaker. | **TRAP**: Using fixed loop counters (e.g. `for i in range(3):`) as normal completion logic. | Prevents the agent from solving tasks that require 4 or 5 iterations, or wastes tokens looping when already finished. |

---

## 💡 Architectural Decision Matrix: Rules of Thumb

| Architectural Dilemma | Recommended Choice | Rapid 30-Second Decision Rule |
|---|---|---|
| **How to decide whether to call more tools?** | Check `stop_reason == "tool_use"` | If `stop_reason == "tool_use"`, execute tools. If `"end_turn"`, return. Never inspect text for loop control. |
| **How to handle sampling with Extended Thinking?** | Omit `temperature` completely | When `thinking={"type": "enabled", "budget_tokens": ...}`, passing `temperature`, `top_p`, or `top_k` causes HTTP 400. |
| **How to choose models for orchestrator loops?** | `claude-sonnet-4-5` for orchestrator, `claude-haiku-4-5` for fast tool tasks | Sonnet provides superior reasoning for multi-step tool sequencing; Haiku provides ultra-fast lightweight execution. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

In Claude 4.5 (`claude-sonnet-4-5` and `claude-haiku-4-5`), extended thinking enables the model to reason through complex multi-step tool decisions before committing to an action:

1. **Configuration**:
   ```python
   response = client.messages.create(
       model="claude-sonnet-4-5",
       max_tokens=4096,
       thinking={"type": "enabled", "budget_tokens": 2048},
       # NOTE: temperature, top_p, and top_k MUST BE OMITTED!
       messages=messages,
       tools=tools
   )
   ```
2. **Preservation Invariant**:
   When Claude replies with thinking blocks:
   ```json
   {
     "role": "assistant",
     "content": [
       {"type": "thinking", "thinking": "Let's first calculate...", "signature": "Eq0B..."},
       {"type": "tool_use", "id": "toolu_01", "name": "calculator", "input": {"expression": "125 * 8"}}
     ]
   }
   ```
   This entire block—including the thinking block and its signature—must remain intact when appending to `messages` for turn 2.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_1_1_multi_tool_agent_loop.ipynb`](ts_1_1_multi_tool_agent_loop.ipynb):
1. **Step 1: Environment & Offline Simulation**: Imports the zero-cost offline `MockClaudeClient` and logger.
2. **Step 2: Safe Tool Definitions**: Demonstrates safe AST expression parsing (avoiding unsafe `eval()`) and structured schemas.
3. **Step 3: The Core Autonomous Agent Loop**: Implements the clean while-loop checking `stop_reason`.
4. **Step 4: Multi-Turn Offline Simulation**: Runs Turn 1 (`calculator`) and Turn 2 (`end_turn`) with complete audit trails.
5. **Step 5: Verification & Safety Assertions**: Verifies that dual history, stop reasons, and error bounds are strictly observed.

In [ ]:
# Step 1: Environment & Offline Simulation Setup
import ast
import json
import operator
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

# Dynamically locate repository root
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Import mock client for 100% offline, zero-cost execution
from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse
from lib.response_logger import ResponseTrail

print(" Repository root loaded:", repo_root)
print(" Offline Mock Claude Client ready (Zero API calls, Zero tokens)")

### 🛠️ Step 2: Safe Tool Implementations & Schemas
We implement two production-grade tools:
1. `calculator`: Evaluates arithmetic expressions safely using Python's `ast` module (avoiding unsafe `eval()`).
2. `web_search`: Simulates targeted knowledge retrieval with pre-indexed fact snippets.

In [ ]:
# Step 2: Tool Logic & JSON Schemas
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def safe_eval_expr(node: ast.AST) -> float:
    """Safely evaluate an arithmetic AST expression without eval()."""
    if isinstance(node, ast.Expression):
        return safe_eval_expr(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](safe_eval_expr(node.left), safe_eval_expr(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](safe_eval_expr(node.operand))
    raise ValueError(f"Unsupported AST node: {type(node).__name__}")

def calculator_tool(expression: str) -> Dict[str, Any]:
    """Execute mathematical expressions safely."""
    try:
        parsed = ast.parse(expression.strip(), mode="eval")
        result = safe_eval_expr(parsed)
        return {"result": int(result) if result.is_integer() else result}
    except Exception as exc:
        return {"isError": True, "error": f"Evaluation error: {exc}"}

def web_search_tool(query: str) -> Dict[str, Any]:
    """Simulate search retrieval from trusted reference data."""
    kb = {
        "speed of light": "The speed of light in vacuum is approximately 299,792,458 meters per second.",
        "tokyo population": "The population of Tokyo metropolitan area is approximately 37,400,000.",
    }
    q = query.lower()
    for key, val in kb.items():
        if key in q or any(w in q for w in key.split()):
            return {"found": True, "results": [{"title": key.title(), "snippet": val}]}
    return {"found": False, "results": [], "message": f"No entries for: {query}"}

TOOLS = [
    {
        "name": "calculator",
        "description": "Calculates the result of a mathematical expression. Input must be arithmetic only.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "Arithmetic expression, e.g. '234 * 18'"}},
            "required": ["expression"],
        },
    },
    {
        "name": "web_search",
        "description": "Searches reference documentation and facts by search query.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Search term or question"}},
            "required": ["query"],
        },
    },
]

def execute_tool(tool_name: str, tool_inputs: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name == "calculator":
        return calculator_tool(tool_inputs.get("expression", ""))
    elif tool_name == "web_search":
        return web_search_tool(tool_inputs.get("query", ""))
    return {"isError": True, "error": f"Unknown tool: {tool_name}"}

print(" Tools and schemas initialized successfully.")

### 🔄 Step 3: The Deterministic Agentic Loop Implementation
The core loop orchestrates conversational turns by inspecting `response.stop_reason`.
- If `stop_reason == "tool_use"`, we execute each tool and feed `tool_result` blocks back.
- If `stop_reason == "end_turn"`, the agent has completed reasoning and returns the answer.

In [ ]:
# Step 3: The Core Autonomous Agent Loop
def run_agentic_loop(prompt: str, client: Any, trail: Optional[ResponseTrail] = None, max_iterations: int = 15) -> str:
    """
    Executes the standard agentic loop with stop_reason lifecycle management.
    """
    messages: List[Dict[str, Any]] = [{"role": "user", "content": prompt}]
    session_id = "sess_sim_01"
    iteration = 0

    print(f"\n=== Starting Agentic Loop for prompt: '{prompt}' ===")

    while iteration < max_iterations:
        iteration += 1
        print(f"\n[Turn {iteration}] Calling Claude API...")

        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            messages=messages,
            tools=TOOLS,
        )

        stop_reason = response.stop_reason
        print(f"[Turn {iteration}] stop_reason: '{stop_reason}'")

        # Check if model wants to invoke tools
        if stop_reason == "tool_use":
            # 1. Append assistant turn with raw tool_use blocks
            assistant_content = [b.model_dump() for b in response.content]
            messages.append({"role": "assistant", "content": assistant_content})

            # 2. Execute tools and collect tool_result blocks
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  -> Executing Tool: {block.name}({block.input})")
                    res = execute_tool(block.name, block.input)
                    print(f"  <- Tool Result: {res}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(res),
                    })

            # 3. Append user turn containing tool results
            messages.append({"role": "user", "content": tool_results})
            continue

        # Deterministic exit signal
        if stop_reason == "end_turn":
            final_text = "".join(b.text for b in response.content if b.type == "text")
            print(f"\n[Turn {iteration}] Task Complete. Final Response: {final_text}")
            return final_text

        raise RuntimeError(f"Unexpected stop_reason: {stop_reason}")

    raise TimeoutError(f"Safety limit of {max_iterations} iterations reached.")

### 🧪 Step 4: Multi-Turn Offline Simulation
We simulate a multi-turn conversation without any live API costs:
- **Turn 1**: Claude decides to invoke `calculator` for `125 * 8`.
- **Turn 2**: Claude receives the calculation `1000` and emits `stop_reason == "end_turn"` with the final text.

In [ ]:
# Step 4: Execute Simulation with MockClaudeClient
# Queue Turn 1: tool_use request
turn1_response = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="tool_use",
            name="calculator",
            tool_id="call_calc_981",
            input_data={"expression": "125 * 8"},
        )
    ],
    stop_reason="tool_use",
)

# Queue Turn 2: end_turn conclusion
turn2_response = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="text",
            text="125 multiplied by 8 equals 1,000.",
        )
    ],
    stop_reason="end_turn",
)

mock_client = MockClaudeClient([turn1_response, turn2_response])
final_output = run_agentic_loop("What is 125 * 8?", mock_client)
print("\nResult verified:", final_output)

### 🔬 Step 5: Automated Verification & Edge Case Checks
We verify that:
1. Arithmetic evaluations are completely protected against injection attacks.
2. The agent loop correctly handles single-turn and multi-turn stops.
3. The conversation history correctly interleaves `tool_use` and `tool_result`.

In [ ]:
# Step 5: Verification Assertions
# 1. Verify AST calculator safety against malicious code injection
safe_res = calculator_tool("5 * 20 + 10")
assert safe_res["result"] == 110, f"Expected 110, got {safe_res}"

injection_attempt = calculator_tool("__import__('os').system('ls')")
assert injection_attempt.get("isError") is True, "Safety filter must reject eval injection!"

# 2. Verify web search functionality
search_res = web_search_tool("speed of light")
assert search_res["found"] is True
assert "299,792,458" in search_res["results"][0]["snippet"]

# 3. Verify final output
assert "1,000" in final_output or "1000" in final_output

print(" All verification assertions passed successfully! 100% offline verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Inspect `response.stop_reason` exclusively. Never inspect response text string patterns to determine if an agent is done.
- **Rule 2**: Both `tool_use` and `tool_result` must be appended in distinct conversational roles (`assistant` then `user`).
- **Rule 3**: `MAX_ITERATIONS` is a fallback safeguard, never the primary stopping mechanism.